(Tutorial_Form_file_pdb_gz)=
# file:pdb.gz

*Reading gzip-compressed PDB data through the shared native parser.*

This experimental Tier-3 input form recognizes string and path-like `.pdb.gz` names,
including uppercase extensions. It preserves the archive and neighboring files.
Each read has its own decoded text buffer, with no decompressed disk file.

:::{admonition} API documentation
:class: dropdown
{func}`molsysmt.basic.convert` and {func}`molsysmt.basic.get`
:::

:::{versionadded} 1.0.0
:::

## Supported input routes

| Target | Result |
| --- | --- |
| `molsysmt.MolSys` | Native topology, chemical declarations and coordinates. |
| `molsysmt.Topology` | Coordinate-independent native hierarchy and connectivity. |
| `molsysmt.Structures` | Coordinates, box, B factors and occupancy with active units. |
| `molsysmt.PDBFileHandler` | Parsed native handler; the caller closes its buffer. |
| `string:pdb_text` | Decoded original text, or an explicitly selected PDB text. |

Public `get` pipes supported PDB attributes through the native handler. Selections
use **atom indices** and **structure indices**, distinct from the file's IDs.

## Reading with explicit connectivity policy

This example constructs a small archive from a bundled PDB. Real `.pdb.gz` inputs
use the same route; compression does not require a different molecular parser.
`get_missing_bonds=False` retains only deposited records, independent of OpenMM.


In [1]:
import gzip
from pathlib import Path
from tempfile import TemporaryDirectory
import molsysmt as msm

source = msm.systems['T4 lysozyme L99A']['181l.pdb']
with TemporaryDirectory() as directory:
    archive = Path(directory) / '181l.pdb.gz'
    payload = Path(source).read_bytes()
    archive.write_bytes(gzip.compress(payload, mtime=0))
    original = archive.read_bytes()
    molsys = msm.convert(archive, to_form='molsysmt.MolSys', get_missing_bonds=False)
    assert msm.get_form(archive) == 'file:pdb.gz'
    assert msm.get(molsys, n_atoms=True, n_bonds=True) == [1441, 13]
    selected = msm.convert(archive, selection=[2, 0, 4], structure_indices=[0],
                           to_form='molsysmt.MolSys', get_missing_bonds=False)
    assert selected.structures.coordinates.shape == (1, 3, 3)
    assert msm.convert(archive, to_form='string:pdb_text') == payload.decode('utf-8')
    assert archive.read_bytes() == original
    assert list(Path(directory).iterdir()) == [archive]
    print('Native counts:', msm.get(molsys, n_atoms=True, n_bonds=True, n_structures=True))
    print('Selected coordinate shape:', selected.structures.coordinates.shape)


Native counts: [1441, 13, 1]
Selected coordinate shape: (1, 3, 3)


## Scientific policy and resource lifetime

The plain and gzip PDB native routes share the existing parser. The default
`get_missing_bonds=True` retains the plain reader's optional OpenMM inference
and diagnosed fallback. Explicit `bond_inference_engine='MolSysMT'` or `'OpenMM'`
retains the existing policy; it requires `get_missing_bonds=True`. Reading PDB
records does not establish complete bond orders, protonation or chemical states.

Length quantities follow the active PyUnitWizard policy. A topology has no
coordinate structure axis; its `structure_indices` option has the same
coordinate-independent semantics as plain PDB topology conversion.

Malformed headers, truncated data, failed CRC or UTF-8 decoding raise
`FileContentError` with filename and original cause before inference. Filesystem
and native parser errors propagate. Read streams and temporary parser buffers close
on success or failure. A handler returned explicitly transfers buffer ownership;
call `handler.close()` after use. Concurrent reads never write next to the source.

## Limits

Native parsing is eager: decoded text and parsed records occupy memory. This
adapter offers input conversion and native attribute delivery; it does not offer
compressed writing, in-place editing, identity round trips or trajectory streaming.
Convert to a supported native form for molecular editing and persistence.

:::{seealso}
:class: dropdown
{ref}`Tutorial_Form_file_pdb` documents plain input.
{ref}`cookbook-pdb-connectivity-policy` explains native connectivity choices.
{ref}`user-foundations-support-forms-files` lists file-form contracts.
:::
